# Sonar Mines vs Rocks Classification

A sonar pulse bounced off a metal cylinder (a mine) returns a different energy pattern across 60 frequency bands than one bounced off a roughly cylindrical rock. With only 208 signals, this classic benchmark asks whether 60 numbers are enough to tell mine from rock.

In [1]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# UCI Connectionist Bench (Sonar, Mines vs. Rocks) (id 151)
df = pd.read_csv(io.BytesIO(download('https://archive.ics.uci.edu/static/public/151/data.csv')))
print(df.shape)
print('Signals per class (M = mine, metal cylinder; R = rock):')
print(df['class'].value_counts())
df.head(3)


(208, 61)
Signals per class (M = mine, metal cylinder; R = rock):
class
M    111
R     97
Name: count, dtype: int64


,Attribute1,Attribute2,Attribute3,Attribute4,Attribute5,Attribute6,Attribute7,Attribute8,Attribute9,Attribute10,...,Attribute52,Attribute53,Attribute54,Attribute55,Attribute56,Attribute57,Attribute58,Attribute59,Attribute60,class
0,0.0200,0.0371,0.0428,0.0207,0.0954,0.0986,0.1539,0.1601,0.3109,0.2111,...,0.0027,0.0065,0.0159,0.0072,0.0167,0.0180,0.0084,0.0090,0.0032,R
1,0.0453,0.0523,0.0843,0.0689,0.1183,0.2583,0.2156,0.3481,0.3337,0.2872,...,0.0084,0.0089,0.0048,0.0094,0.0191,0.0140,0.0049,0.0052,0.0044,R
2,0.0262,0.0582,0.1099,0.1083,0.0974,0.2280,0.2431,0.3771,0.5598,0.6194,...,0.0232,0.0166,0.0095,0.0180,0.0244,0.0316,0.0164,0.0095,0.0078,R


In [2]:

feats = [c for c in df.columns if c != 'class']
X = df[feats]; y = df['class']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
acc_rf = accuracy_score(yte, rf.predict(Xte)); acc_lr = accuracy_score(yte, lr.predict(Xte))
print(f'Random Forest accuracy: {acc_rf:.4f}')
print(f'Logistic Regression accuracy: {acc_lr:.4f}')
print(classification_report(yte, rf.predict(Xte)))


Random Forest accuracy: 0.8654
Logistic Regression accuracy: 0.8269
              precision    recall  f1-score   support

           M       0.80      1.00      0.89        28
           R       1.00      0.71      0.83        24

    accuracy                           0.87        52
   macro avg       0.90      0.85      0.86        52
weighted avg       0.89      0.87      0.86        52



In [3]:
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print(imp)
imp.plot(kind='barh', title='Which frequency bands separate mines from rocks?')
plt.tight_layout()
plt.show()
winner = rf if acc_rf >= acc_lr else lr
winner_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
print(f'Winner: {winner_name}')
cm = confusion_matrix(yte, winner.predict(Xte), labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y.unique()))
disp.plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {winner_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


Attribute12    0.042864
Attribute10    0.042175
Attribute51    0.039812
Attribute48    0.030603
Attribute21    0.030601
Attribute11    0.029845
Attribute9     0.027895
Attribute52    0.027709
Attribute4     0.027705
Attribute47    0.027178
Attribute13    0.026944
Attribute36    0.025900
Attribute20    0.023143
Attribute45    0.021144
Attribute44    0.021044
Attribute49    0.019053
Attribute46    0.018760
Attribute16    0.018040
Attribute37    0.017342
Attribute17    0.017025
Attribute39    0.016966
Attribute27    0.016041
Attribute28    0.015813
Attribute6     0.015255
Attribute33    0.015085
Attribute31    0.014702
Attribute1     0.014644
Attribute34    0.014600
Attribute43    0.014147
Attribute18    0.014028
Attribute22    0.014003
Attribute15    0.013768
Attribute5     0.013417
Attribute14    0.012716
Attribute23    0.012577
Attribute30    0.012434
Attribute42    0.012344
Attribute19    0.012286
Attribute35    0.012271
Attribute54    0.012268
Attribute56    0.012261
Attribute55    0

Winner: Random Forest
